In [1]:
from pyspark.sql import SparkSession

In [63]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("PySpark_Hackathon")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

sc = spark.sparkContext

print("Master:", sc.master)
print("Driver host:", sc.getConf().get("spark.driver.host"))
print("Bind address:", sc.getConf().get("spark.driver.bindAddress"))
print("Python:", sys.executable)

Master: local[2]
Driver host: 127.0.0.1
Bind address: 127.0.0.1
Python: C:\Users\Lenovo\anaconda3\python.exe


In [134]:
data = [
    (1, "Aman", "CSE", 85),
    (2, "Priya", "IT", 92),
    (3, "Rahul", "CSE", 67),
    (4, "Neha", "ECE", 78),
    (5, "Ravi", "IT", 55)
]

columns = ["id", "name", "department", "marks"]

df = spark.createDataFrame(data, columns)
df.show()

+---+-----+----------+-----+
| id| name|department|marks|
+---+-----+----------+-----+
|  1| Aman|       CSE|   85|
|  2|Priya|        IT|   92|
|  3|Rahul|       CSE|   67|
|  4| Neha|       ECE|   78|
|  5| Ravi|        IT|   55|
+---+-----+----------+-----+



In [6]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- marks: long (nullable = true)



In [8]:
df.columns

['id', 'name', 'department', 'marks']

In [9]:
df.count()

5

In [10]:
df.select("name","marks").show()

+-----+-----+
| name|marks|
+-----+-----+
| Aman|   85|
|Priya|   92|
|Rahul|   67|
| Neha|   78|
| Ravi|   55|
+-----+-----+



In [11]:
df.filter(df['marks']>80).show()

+---+-----+----------+-----+
| id| name|department|marks|
+---+-----+----------+-----+
|  1| Aman|       CSE|   85|
|  2|Priya|        IT|   92|
+---+-----+----------+-----+



In [12]:
df.where(df['department']=='CSE').show()

+---+-----+----------+-----+
| id| name|department|marks|
+---+-----+----------+-----+
|  1| Aman|       CSE|   85|
|  3|Rahul|       CSE|   67|
+---+-----+----------+-----+



In [14]:
df.orderBy("marks").show()

+---+-----+----------+-----+
| id| name|department|marks|
+---+-----+----------+-----+
|  5| Ravi|        IT|   55|
|  3|Rahul|       CSE|   67|
|  4| Neha|       ECE|   78|
|  1| Aman|       CSE|   85|
|  2|Priya|        IT|   92|
+---+-----+----------+-----+



In [16]:
df.orderBy("marks",ascending=False).show()

+---+-----+----------+-----+
| id| name|department|marks|
+---+-----+----------+-----+
|  2|Priya|        IT|   92|
|  1| Aman|       CSE|   85|
|  4| Neha|       ECE|   78|
|  3|Rahul|       CSE|   67|
|  5| Ravi|        IT|   55|
+---+-----+----------+-----+



In [17]:
from pyspark.sql.functions import desc
df.orderBy(desc("marks")).show()

+---+-----+----------+-----+
| id| name|department|marks|
+---+-----+----------+-----+
|  2|Priya|        IT|   92|
|  1| Aman|       CSE|   85|
|  4| Neha|       ECE|   78|
|  3|Rahul|       CSE|   67|
|  5| Ravi|        IT|   55|
+---+-----+----------+-----+



In [7]:
from pyspark.sql.functions import col,when

df = df.withColumn(
    "status", when(col("marks")>75,"Passed").otherwise("Failed")
)
df.show()

+---+-----+----------+-----+------+
| id| name|department|marks|status|
+---+-----+----------+-----+------+
|  1| Aman|       CSE|   85|Passed|
|  2|Priya|        IT|   92|Passed|
|  3|Rahul|       CSE|   67|Failed|
|  4| Neha|       ECE|   78|Passed|
|  5| Ravi|        IT|   55|Failed|
+---+-----+----------+-----+------+



In [8]:
df.filter(
    (col("marks")>75) & (col("department")=='CSE')
).show()

+---+----+----------+-----+------+
| id|name|department|marks|status|
+---+----+----------+-----+------+
|  1|Aman|       CSE|   85|Passed|
+---+----+----------+-----+------+



In [11]:
df.filter(
    (col("marks")>80) & (col("department")=='IT')
).show()

+---+-----+----------+-----+------+
| id| name|department|marks|status|
+---+-----+----------+-----+------+
|  2|Priya|        IT|   92|Passed|
+---+-----+----------+-----+------+



In [13]:
df.filter(
    col("department") != 'CSE'
).show()

+---+-----+----------+-----+------+
| id| name|department|marks|status|
+---+-----+----------+-----+------+
|  2|Priya|        IT|   92|Passed|
|  4| Neha|       ECE|   78|Passed|
|  5| Ravi|        IT|   55|Failed|
+---+-----+----------+-----+------+



In [15]:
df = df.withColumn(
    "bonus_marks",
    col("marks")+5
)
df.show()

+---+-----+----------+-----+------+-----------+
| id| name|department|marks|status|bonus_marks|
+---+-----+----------+-----+------+-----------+
|  1| Aman|       CSE|   85|Passed|         90|
|  2|Priya|        IT|   92|Passed|         97|
|  3|Rahul|       CSE|   67|Failed|         72|
|  4| Neha|       ECE|   78|Passed|         83|
|  5| Ravi|        IT|   55|Failed|         60|
+---+-----+----------+-----+------+-----------+



In [23]:
df = df.withColumn(
    "percentage",
    col("marks").cast("double")
)
df.show()

+---+-----+----------+-----+------+-----------+----------+
| id| name|department|marks|status|bonus_marks|percentage|
+---+-----+----------+-----+------+-----------+----------+
|  1| Aman|       CSE|   85|Passed|         90|      85.0|
|  2|Priya|        IT|   92|Passed|         97|      92.0|
|  3|Rahul|       CSE|   67|Failed|         72|      67.0|
|  4| Neha|       ECE|   78|Passed|         83|      78.0|
|  5| Ravi|        IT|   55|Failed|         60|      55.0|
+---+-----+----------+-----+------+-----------+----------+



In [24]:
df.withColumnRenamed("marks","total_marks").show()

+---+-----+----------+-----------+------+-----------+----------+
| id| name|department|total_marks|status|bonus_marks|percentage|
+---+-----+----------+-----------+------+-----------+----------+
|  1| Aman|       CSE|         85|Passed|         90|      85.0|
|  2|Priya|        IT|         92|Passed|         97|      92.0|
|  3|Rahul|       CSE|         67|Failed|         72|      67.0|
|  4| Neha|       ECE|         78|Passed|         83|      78.0|
|  5| Ravi|        IT|         55|Failed|         60|      55.0|
+---+-----+----------+-----------+------+-----------+----------+



In [25]:
df.drop("department").show()

+---+-----+-----+------+-----------+----------+
| id| name|marks|status|bonus_marks|percentage|
+---+-----+-----+------+-----------+----------+
|  1| Aman|   85|Passed|         90|      85.0|
|  2|Priya|   92|Passed|         97|      92.0|
|  3|Rahul|   67|Failed|         72|      67.0|
|  4| Neha|   78|Passed|         83|      78.0|
|  5| Ravi|   55|Failed|         60|      55.0|
+---+-----+-----+------+-----------+----------+



In [32]:
from pyspark.sql.functions import ( count,sum,avg,min,max )
df.agg(
    count("*").alias("total_students"),
    sum("marks").alias("total_marks"),
    avg("marks").alias("average_marks"),
    min("marks").alias("minimum_marks"),
    max("marks").alias("maximum_marks")
).show()

+--------------+-----------+-------------+-------------+-------------+
|total_students|total_marks|average_marks|minimum_marks|maximum_marks|
+--------------+-----------+-------------+-------------+-------------+
|             5|        377|         75.4|           55|           92|
+--------------+-----------+-------------+-------------+-------------+



In [28]:
df.count()

5

In [34]:
df.groupBy("department").agg(count("*").alias("student_count")).show()

+----------+-------------+
|department|student_count|
+----------+-------------+
|        IT|            2|
|       CSE|            2|
|       ECE|            1|
+----------+-------------+



In [36]:
df.groupBy("department").agg(
    count("*").alias("student_count"),
    avg("marks").alias("average_marks"),
    max("marks").alias("highest_marks")
).show()

+----------+-------------+-------------+-------------+
|department|student_count|average_marks|highest_marks|
+----------+-------------+-------------+-------------+
|        IT|            2|         73.5|           92|
|       CSE|            2|         76.0|           85|
|       ECE|            1|         78.0|           78|
+----------+-------------+-------------+-------------+



In [38]:
sample_data = [
    (1, "Aman", 85),
    (2, None, 92),
    (3, "Rahul", None),
    (4, "Aman", 85)
]

df2 = spark.createDataFrame(
    sample_data,
    ["id", "name", "marks"]
)

df2.show()

+---+-----+-----+
| id| name|marks|
+---+-----+-----+
|  1| Aman|   85|
|  2| NULL|   92|
|  3|Rahul| NULL|
|  4| Aman|   85|
+---+-----+-----+



In [39]:
df2.filter(
    col("name").isNull()
).show()

+---+----+-----+
| id|name|marks|
+---+----+-----+
|  2|NULL|   92|
+---+----+-----+



In [43]:
df2.dropna().show()

+---+----+-----+
| id|name|marks|
+---+----+-----+
|  1|Aman|   85|
|  4|Aman|   85|
+---+----+-----+



In [44]:
df2 = df2.fillna({"name":"unknown"})
df2.show()

+---+-------+-----+
| id|   name|marks|
+---+-------+-----+
|  1|   Aman|   85|
|  2|unknown|   92|
|  3|  Rahul| NULL|
|  4|   Aman|   85|
+---+-------+-----+



In [45]:
df2 = df2.fillna({"marks":00})
df2.show()

+---+-------+-----+
| id|   name|marks|
+---+-------+-----+
|  1|   Aman|   85|
|  2|unknown|   92|
|  3|  Rahul|    0|
|  4|   Aman|   85|
+---+-------+-----+



In [46]:
df2.dropDuplicates().show()

+---+-------+-----+
| id|   name|marks|
+---+-------+-----+
|  2|unknown|   92|
|  1|   Aman|   85|
|  4|   Aman|   85|
|  3|  Rahul|    0|
+---+-------+-----+



In [48]:
df2.select("name").dropDuplicates().show()
df2.show()

+-------+
|   name|
+-------+
|unknown|
|   Aman|
|  Rahul|
+-------+

+---+-------+-----+
| id|   name|marks|
+---+-------+-----+
|  1|   Aman|   85|
|  2|unknown|   92|
|  3|  Rahul|    0|
|  4|   Aman|   85|
+---+-------+-----+



In [49]:
df2.select("name").distinct().show()

+-------+
|   name|
+-------+
|unknown|
|   Aman|
|  Rahul|
+-------+



In [51]:
students = spark.createDataFrame(
    [
        (1, "Aman", 101),
        (2, "Priya", 102),
        (3, "Rahul", 101),
        (4, "Neha", 103)
    ],
    ["student_id", "name", "dept_id"]
)

departments = spark.createDataFrame(
    [
        (101, "CSE"),
        (102, "IT"),
        (104, "ECE")
    ],
    ["dept_id", "department"]
)
students.show()
departments.show()

+----------+-----+-------+
|student_id| name|dept_id|
+----------+-----+-------+
|         1| Aman|    101|
|         2|Priya|    102|
|         3|Rahul|    101|
|         4| Neha|    103|
+----------+-----+-------+

+-------+----------+
|dept_id|department|
+-------+----------+
|    101|       CSE|
|    102|        IT|
|    104|       ECE|
+-------+----------+



In [53]:
students.join(
    departments,
    on="dept_id",
    how = "inner"
).show()

+-------+----------+-----+----------+
|dept_id|student_id| name|department|
+-------+----------+-----+----------+
|    101|         1| Aman|       CSE|
|    101|         3|Rahul|       CSE|
|    102|         2|Priya|        IT|
+-------+----------+-----+----------+



In [55]:
students.join(
    departments,
    on="dept_id",
    how= "left"
).show()

+-------+----------+-----+----------+
|dept_id|student_id| name|department|
+-------+----------+-----+----------+
|    101|         1| Aman|       CSE|
|    102|         2|Priya|        IT|
|    103|         4| Neha|      NULL|
|    101|         3|Rahul|       CSE|
+-------+----------+-----+----------+



In [59]:
students.join(
    departments,
    on="dept_id",
    how="left_semi"
).show()

+-------+----------+-----+
|dept_id|student_id| name|
+-------+----------+-----+
|    101|         1| Aman|
|    101|         3|Rahul|
|    102|         2|Priya|
+-------+----------+-----+



In [60]:
df.createOrReplaceTempView("students")

In [65]:
spark.sql(
    """
    Select name , marks from students where marks > 72 order by marks desc
    """
).show()

+-----+-----+
| name|marks|
+-----+-----+
|Priya|   92|
| Aman|   85|
| Neha|   78|
+-----+-----+



In [75]:
spark.sql("""
    select department, avg(marks) as average_marks,
    count(*) as total_students
    from students 
    group by department
""").show()

+----------+-------------+--------------+
|department|average_marks|total_students|
+----------+-------------+--------------+
|        IT|         73.5|             2|
|       CSE|         76.0|             2|
|       ECE|         78.0|             1|
+----------+-------------+--------------+



In [81]:
spark.sql("""
    select marks as second_highest 
    from students
    order by marks desc limit 1 offset 1
""").show()

+--------------+
|second_highest|
+--------------+
|            85|
+--------------+



In [83]:
import pandas as pd

pdf = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["Aman", "Priya", "Rahul", "Neha", "Ravi"],
    "department": ["CSE", "IT", "CSE", "ECE", "IT"],
    "marks": [85, 92, 67, 78, 55]
})

csv_path = r"C:\Users\Lenovo\OneDrive\Desktop\PYTHON NOTEBOOK\anaconda_projects\students.csv"

pdf.to_csv(csv_path, index=False)
print("CSV created:", csv_path)

CSV created: C:\Users\Lenovo\OneDrive\Desktop\PYTHON NOTEBOOK\anaconda_projects\students.csv


In [87]:
csv_df = (
    spark.read
    .option("header",True)
    .option("inferschema",True)
    .csv(r"C:\Users\Lenovo\OneDrive\Desktop\PYTHON NOTEBOOK\anaconda_projects\students.csv")
)
csv_df.show()
csv_df.printSchema()

+---+-----+----------+-----+
| id| name|department|marks|
+---+-----+----------+-----+
|  1| Aman|       CSE|   85|
|  2|Priya|        IT|   92|
|  3|Rahul|       CSE|   67|
|  4| Neha|       ECE|   78|
|  5| Ravi|        IT|   55|
+---+-----+----------+-----+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- marks: integer (nullable = true)



In [97]:
number = [10,20,30,40,60]
rdd = spark.sparkContext.parallelize(number)
print(rdd.collect())
print("total length are : ", rdd.count())

[10, 20, 30, 40, 60]
total length are :  5


In [100]:
square_rdd = rdd.map(lambda x : x * x)
print(square_rdd.collect())

[100, 400, 900, 1600, 3600]


In [109]:
cube_rdd = rdd.map(lambda x : x*10)
print(cube_rdd.collect())

[100, 200, 300, 400, 600]


In [111]:
even = rdd.filter(lambda x : x%2==0)
print(even.collect())

[10, 20, 30, 40, 60]


In [112]:
print(rdd.sum())

160


In [114]:
df[df['marks']<70].show()

+---+-----+----------+-----+------+-----------+----------+
| id| name|department|marks|status|bonus_marks|percentage|
+---+-----+----------+-----+------+-----------+----------+
|  3|Rahul|       CSE|   67|Failed|         72|      67.0|
|  5| Ravi|        IT|   55|Failed|         60|      55.0|
+---+-----+----------+-----+------+-----------+----------+



In [117]:
df.groupby('department').avg("marks").show()

+----------+----------+
|department|avg(marks)|
+----------+----------+
|        IT|      73.5|
|       CSE|      76.0|
|       ECE|      78.0|
+----------+----------+



In [119]:
spark.sql("""
    select name,marks 
    from students
    order by marks desc limit 3 
""").show()

+-----+-----+
| name|marks|
+-----+-----+
|Priya|   92|
| Aman|   85|
| Neha|   78|
+-----+-----+



In [135]:
df = df.withColumn(
    "updated_status",
    when(col('marks')>80,"Excellent")
    .otherwise("Good")
)
df.show()

+---+-----+----------+-----+--------------+
| id| name|department|marks|updated_status|
+---+-----+----------+-----+--------------+
|  1| Aman|       CSE|   85|     Excellent|
|  2|Priya|        IT|   92|     Excellent|
|  3|Rahul|       CSE|   67|          Good|
|  4| Neha|       ECE|   78|          Good|
|  5| Ravi|        IT|   55|          Good|
+---+-----+----------+-----+--------------+



In [136]:
df.show()

+---+-----+----------+-----+--------------+
| id| name|department|marks|updated_status|
+---+-----+----------+-----+--------------+
|  1| Aman|       CSE|   85|     Excellent|
|  2|Priya|        IT|   92|     Excellent|
|  3|Rahul|       CSE|   67|          Good|
|  4| Neha|       ECE|   78|          Good|
|  5| Ravi|        IT|   55|          Good|
+---+-----+----------+-----+--------------+



In [137]:
import os

print("Current folder:", os.getcwd())

print(
    "Notebook folder:",
    os.path.abspath("Pyspark_first_learning.ipynb")
)

Current folder: C:\Users\Lenovo\anaconda_projects
Notebook folder: C:\Users\Lenovo\anaconda_projects\Pyspark_first_learning.ipynb
